# Unsupervised feature-selection experiment on combined features

This notebook compares the full 72-feature combined representation with Correlation Filtering, Variance Threshold, PCA, and Laplacian Score for K-Means, DBSCAN, Agglomerative Hierarchical Clustering, and Gaussian Mixture Models. Feature selection is fitted using training inputs only. Binary labels are used only after clustering for external evaluation and cluster-to-label mapping.

Run names follow `Combined_{feature_selection}_{algorithm}`, for example `Combined_variancethreshold_K-means`.

In [1]:
import platform
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csgraph
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, calinski_harabasz_score,
    completeness_score, davies_bouldin_score, f1_score, homogeneity_score,
    normalized_mutual_info_score, precision_score, recall_score,
    silhouette_score, v_measure_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from configs.config import EXPERIMENT_NAME, RANDOM_STATE
TRACKING_DB = (PROJECT_ROOT / "Notebooks" / "mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB.as_posix()}")
mlflow.set_experiment(EXPERIMENT_NAME)

CLUSTER_TRAIN_SIZE = 8_000
CLUSTER_TEST_SIZE = 4_000
SELECTION_SAMPLE_SIZE = 4_000
CORRELATION_THRESHOLD = 0.90
VARIANCE_THRESHOLD = 0.01
PCA_VARIANCE_TO_KEEP = 0.95
LAPLACIAN_TOP_K = 36
LAPLACIAN_NEIGHBORS = 10
INTERNAL_SAMPLE_SIZE = 3_000


In [2]:
data_path = PROJECT_ROOT / "Data" / "Consolidated_df.csv"
df = pd.read_csv(data_path)
ORIGINAL_FEATURES = (
    "duration", "protocoltype", "service", "flag", "srcbytes", "dstbytes",
    "land", "wrongfragment", "urgent", "hot", "numfailedlogins",
    "loggedin", "numcompromised", "rootshell", "suattempted", "numroot",
    "numfilecreations", "numshells", "numaccessfiles", "numoutboundcmds",
    "ishostlogin", "isguestlogin", "count", "srvcount", "serrorrate",
    "srvserrorrate", "rerrorrate", "srvrerrorrate", "samesrvrate",
    "diffsrvrate", "srvdiffhostrate", "dsthostcount", "dsthostsrvcount",
    "dsthostsamesrvrate", "dsthostdiffsrvrate", "dsthostsamesrcportrate",
    "dsthostsrvdiffhostrate", "dsthostserrorrate", "dsthostsrvserrorrate",
    "dsthostrerrorrate", "dsthostsrvrerrorrate",
)
ENGINEERED_FEATURES = (
    "total_bytes", "bytes_per_second", "src_dst_byte_ratio",
    "src_byte_fraction", "dst_byte_fraction", "byte_asymmetry",
    "service_connection_ratio", "host_service_ratio",
    "different_service_connections", "different_host_service_connections",
    "short_term_scan_pressure", "host_scan_pressure", "same_source_port_pressure",
    "short_term_serror_score", "short_term_rerror_score", "short_term_error_score",
    "host_serror_score", "host_rerror_score", "host_error_score",
    "same_service_rate_gap", "different_service_rate_gap",
    "serror_rate_gap", "rerror_rate_gap", "authentication_risk_score",
    "has_failed_login", "failed_login_and_logged_in",
    "suspicious_admin_activity", "privileged_activity_score",
    "content_risk_score", "file_and_shell_activity", "root_compromise_ratio",
)
COMBINED_FEATURES = ORIGINAL_FEATURES + ENGINEERED_FEATURES
X = df[list(COMBINED_FEATURES)]
y = df["binary_target"].map({"Normal": 0, "Attack": 1})
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
def sample_frame(X_frame, y_series, size, seed):
    idx, _ = train_test_split(
        np.arange(len(X_frame)), train_size=min(size, len(X_frame)-1),
        stratify=y_series, random_state=seed,
    )
    return X_frame.iloc[idx].copy(), y_series.iloc[idx].copy()
X_cluster_train, y_cluster_train = sample_frame(X_train, y_train, CLUSTER_TRAIN_SIZE, RANDOM_STATE)
X_cluster_test, y_cluster_test = sample_frame(X_test, y_test, CLUSTER_TEST_SIZE, RANDOM_STATE + 1)
categorical_features = X_cluster_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_features = [f for f in COMBINED_FEATURES if f not in categorical_features]
print(f"Combined features={len(COMBINED_FEATURES)}; train sample={len(X_cluster_train)}; test sample={len(X_cluster_test)}")

Combined features=72; train sample=8000; test sample=4000


In [3]:
selector_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", MinMaxScaler()),
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ("scale", MinMaxScaler()),
    ]), categorical_features),
], remainder="drop", verbose_feature_names_out=False)
selection_idx, _ = train_test_split(
    np.arange(len(X_cluster_train)), train_size=min(SELECTION_SAMPLE_SIZE, len(X_cluster_train)-1),
    stratify=y_cluster_train, random_state=RANDOM_STATE,
)
X_selection_numeric = selector_preprocessor.fit_transform(X_cluster_train.iloc[selection_idx])
selector_feature_names = numeric_features + categorical_features
assert X_selection_numeric.shape[1] == len(selector_feature_names)

correlation = pd.DataFrame(X_selection_numeric, columns=selector_feature_names).corr().abs()
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
correlation_removed = [column for column in upper.columns if any(upper[column] > CORRELATION_THRESHOLD)]
correlation_features = [f for f in COMBINED_FEATURES if f not in correlation_removed]

variance_selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
variance_selector.fit(X_selection_numeric)
variance_features = [f for f, keep in zip(selector_feature_names, variance_selector.get_support()) if keep]

W = kneighbors_graph(
    X_selection_numeric, n_neighbors=LAPLACIAN_NEIGHBORS, mode="distance", include_self=False
)
positive_distances = W.data[W.data > 0]
heat_scale = float(np.median(positive_distances)) if len(positive_distances) else 1.0
W.data = np.exp(-(W.data ** 2) / (2 * heat_scale ** 2))
W = (W + W.T) * 0.5
degree = np.asarray(W.sum(axis=1)).ravel()
laplacian = csgraph.laplacian(W, normed=False)
laplacian_scores = []
for feature_index in range(X_selection_numeric.shape[1]):
    values = X_selection_numeric[:, feature_index].astype(float)
    weighted_mean = np.dot(degree, values) / max(degree.sum(), 1e-12)
    centered = values - weighted_mean
    numerator = float(centered @ (laplacian @ centered))
    denominator = float(np.dot(degree, centered ** 2))
    laplacian_scores.append(numerator / denominator if denominator > 1e-12 else np.inf)
laplacian_order = np.argsort(laplacian_scores)
laplacian_features = [selector_feature_names[i] for i in laplacian_order[:min(LAPLACIAN_TOP_K, len(selector_feature_names))]]

def make_raw_preprocessor(features):
    nums = [f for f in features if f in numeric_features]
    cats = [f for f in features if f in categorical_features]
    transformers = []
    if nums: transformers.append(("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
    ]), nums))
    if cats: transformers.append(("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cats))
    return ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)

representations = {}
selection_definitions = {
    "none": list(COMBINED_FEATURES),
    "correlationfiltering": correlation_features,
    "variancethreshold": variance_features,
    "laplacianscore": laplacian_features,
}
for method, features in selection_definitions.items():
    transformer = make_raw_preprocessor(features)
    train_values = transformer.fit_transform(X_cluster_train[features])
    test_values = transformer.transform(X_cluster_test[features])
    representations[method] = {
        "train": train_values, "test": test_values, "transformer": transformer,
        "selected_features": features, "output_dimensions": train_values.shape[1],
    }
full_preprocessor = make_raw_preprocessor(list(COMBINED_FEATURES))
full_train_values = full_preprocessor.fit_transform(X_cluster_train)
full_test_values = full_preprocessor.transform(X_cluster_test)
pca_selector = PCA(n_components=PCA_VARIANCE_TO_KEEP, svd_solver="full", random_state=RANDOM_STATE)
pca_train = pca_selector.fit_transform(full_train_values)
pca_test = pca_selector.transform(full_test_values)
representations["pca"] = {
    "train": pca_train, "test": pca_test,
    "transformer": Pipeline([("preprocessor", full_preprocessor), ("pca", pca_selector)]),
    "selected_features": [f"PC{i+1}" for i in range(pca_train.shape[1])],
    "output_dimensions": pca_train.shape[1],
}
selection_details = {
    "none": {"method": "none"},
    "correlationfiltering": {"threshold": CORRELATION_THRESHOLD, "removed_features": correlation_removed},
    "variancethreshold": {"threshold": VARIANCE_THRESHOLD, "variances": {f: float(v) for f, v in zip(selector_feature_names, variance_selector.variances_)}},
    "laplacianscore": {"top_k": LAPLACIAN_TOP_K, "neighbors": LAPLACIAN_NEIGHBORS, "scores": {f: float(v) for f, v in zip(selector_feature_names, laplacian_scores)}},
    "pca": {"variance_to_keep": PCA_VARIANCE_TO_KEEP, "explained_variance": float(pca_selector.explained_variance_ratio_.sum())},
}
representation_summary_df = pd.DataFrame([
    {"feature_selection": method, "raw_selected_count": len(record["selected_features"]), "output_dimensions": record["output_dimensions"]}
    for method, record in representations.items()
]).sort_values("output_dimensions")
representation_summary_df

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


,feature_selection,raw_selected_count,output_dimensions
4,pca,29,29
3,laplacianscore,36,48
2,variancethreshold,40,114
1,correlationfiltering,51,125
0,none,72,146


In [4]:
def fit_mapping(cluster_ids, labels):
    clusters, classes = np.unique(cluster_ids), np.unique(labels)
    table = np.array([[np.sum((cluster_ids == c) & (labels == y)) for y in classes] for c in clusters])
    rows, cols = linear_sum_assignment(-table)
    mapping = {int(clusters[r]): int(classes[c]) for r, c in zip(rows, cols)}
    default = int(pd.Series(labels).mode().iloc[0])
    for cluster in clusters:
        if int(cluster) not in mapping:
            mapping[int(cluster)] = int(pd.Series(labels[cluster_ids == cluster]).mode().iloc[0])
    return mapping, default
def map_clusters(ids, mapping, default): return np.array([mapping.get(int(c), default) for c in ids])
def predict_dbscan(model, values):
    if len(model.core_sample_indices_) == 0: return np.full(len(values), -1, dtype=int)
    labels = model.labels_[model.core_sample_indices_]
    nn = NearestNeighbors(n_neighbors=1).fit(model.components_)
    distance, index = nn.kneighbors(values)
    pred = labels[index[:, 0]].astype(int); pred[distance[:, 0] > model.eps] = -1
    return pred
def nearest_centroid(test, train, labels):
    ids = np.unique(labels); centers = np.vstack([train[labels == c].mean(axis=0) for c in ids])
    dist = ((test[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    return ids[np.argmin(dist, axis=1)]
def internal_metrics(values, clusters):
    if len(np.unique(clusters)) < 2: return {"silhouette_score": np.nan, "davies_bouldin_index": np.nan, "calinski_harabasz_index": np.nan}
    return {
        "silhouette_score": silhouette_score(values, clusters, sample_size=min(INTERNAL_SAMPLE_SIZE, len(values)), random_state=RANDOM_STATE),
        "davies_bouldin_index": davies_bouldin_score(values, clusters),
        "calinski_harabasz_index": calinski_harabasz_score(values, clusters),
    }
def evaluate(values, clusters, labels, mapped):
    result = internal_metrics(values, clusters)
    result.update({
        "adjusted_rand_index": adjusted_rand_score(labels, clusters),
        "normalized_mutual_information": normalized_mutual_info_score(labels, clusters),
        "homogeneity": homogeneity_score(labels, clusters),
        "completeness": completeness_score(labels, clusters),
        "v_measure": v_measure_score(labels, clusters),
        "mapped_accuracy": accuracy_score(labels, mapped),
        "mapped_precision": precision_score(labels, mapped, zero_division=0),
        "mapped_recall": recall_score(labels, mapped, zero_division=0),
        "mapped_f1": f1_score(labels, mapped, zero_division=0),
        "cluster_count": int(len(np.unique(clusters))), "noise_ratio": float(np.mean(clusters == -1)),
    }); return result
ALGORITHMS = ["K-means", "DBSCAN", "AgglomerativeHierarchical", "GaussianMixtureModel"]
def fit_algorithm(name, train, test):
    if name == "K-means":
        model = KMeans(n_clusters=2, n_init=20, max_iter=500, random_state=RANDOM_STATE); model.fit(train); return model, model.predict(train), model.predict(test), {"n_clusters": 2, "n_init": 20, "max_iter": 500}
    if name == "DBSCAN":
        distances, _ = NearestNeighbors(n_neighbors=10).fit(train).kneighbors(train)
        eps = float(np.quantile(distances[:, -1], 0.90)); model = DBSCAN(eps=eps, min_samples=10, n_jobs=-1); tr = model.fit_predict(train); return model, tr, predict_dbscan(model, test), {"eps": eps, "min_samples": 10}
    if name == "AgglomerativeHierarchical":
        model = AgglomerativeClustering(n_clusters=2, linkage="ward"); tr = model.fit_predict(train); return model, tr, nearest_centroid(test, train, tr), {"n_clusters": 2, "linkage": "ward"}
    model = GaussianMixture(n_components=2, covariance_type="diag", n_init=5, max_iter=300, random_state=RANDOM_STATE); model.fit(train); return model, model.predict(train), model.predict(test), {"n_components": 2, "covariance_type": "diag", "n_init": 5, "max_iter": 300}


## Train and track 20 feature-selection/clustering combinations

Each representation is evaluated on the same stratified train/test samples. Cluster-to-label mappings are learned from training labels and applied unchanged to test clusters.

In [5]:
train_tracking = X_cluster_train.copy(); train_tracking["binary_target"] = y_cluster_train.map({0: "Normal", 1: "Attack"})
test_tracking = X_cluster_test.copy(); test_tracking["binary_target"] = y_cluster_test.map({0: "Normal", 1: "Attack"})
train_dataset = mlflow.data.from_pandas(train_tracking, source=str(data_path.resolve()), targets="binary_target", name="combined_unsupervised_train")
test_dataset = mlflow.data.from_pandas(test_tracking, source=str(data_path.resolve()), targets="binary_target", name="combined_unsupervised_test")
results = []
for feature_selection, representation in representations.items():
    train_values, test_values = representation["train"], representation["test"]
    for algorithm in ALGORITHMS:
        run_name = f"Combined_{feature_selection}_{algorithm}"
        started = time.perf_counter()
        model, train_clusters, test_clusters, algorithm_params = fit_algorithm(algorithm, train_values, test_values)
        fit_seconds = time.perf_counter() - started
        mapping, default = fit_mapping(train_clusters, y_cluster_train.to_numpy())
        mapped_test = map_clusters(test_clusters, mapping, default)
        metrics = evaluate(test_values, test_clusters, y_cluster_test.to_numpy(), mapped_test)
        metrics["fit_seconds"] = fit_seconds
        with mlflow.start_run(run_name=run_name) as run:
            mlflow.set_tags({
                "task": "unsupervised_feature_selection", "data_variant": "Combined",
                "feature_selection": feature_selection, "algorithm": algorithm,
                "labels_used_for_training": "false",
            })
            mlflow.log_input(train_dataset, context="clustering_training"); mlflow.log_input(test_dataset, context="evaluation")
            mlflow.log_params({
                **algorithm_params, "algorithm": algorithm, "feature_selection": feature_selection,
                "input_raw_features": len(COMBINED_FEATURES),
                "selected_raw_features": len(representation["selected_features"]),
                "representation_dimensions": representation["output_dimensions"],
                "dimension_reduction_percent": 100 * (1 - representation["output_dimensions"] / representations["none"]["output_dimensions"]),
                "cluster_train_rows": len(X_cluster_train), "cluster_test_rows": len(X_cluster_test),
                "selection_sample_rows": len(selection_idx), "random_state": RANDOM_STATE,
            })
            mlflow.log_metrics({k: float(v) for k, v in metrics.items() if np.isfinite(v)})
            mlflow.log_dict({
                "selected_features_or_components": representation["selected_features"],
                "selection_details": selection_details[feature_selection],
                "cluster_to_binary_label": {str(k): int(v) for k, v in mapping.items()},
                "default_label": default,
            }, "feature_selection/representation_metadata.json")
            mlflow.log_dict({
                "original_features": list(ORIGINAL_FEATURES), "engineered_features": list(ENGINEERED_FEATURES),
                "sampling_reason": "common sample required by quadratic clustering methods",
                "python": platform.python_version(), "sklearn": sklearn.__version__,
            }, "metadata/run_metadata.json")
            visual = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(test_values) if test_values.shape[1] > 2 else test_values[:, :2]
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            axes[0].scatter(visual[:, 0], visual[:, 1], c=test_clusters, s=5, cmap="tab20"); axes[0].set_title(f"{algorithm} clusters")
            axes[1].scatter(visual[:, 0], visual[:, 1], c=y_cluster_test, s=5, cmap="coolwarm"); axes[1].set_title("True labels (evaluation only)")
            fig.tight_layout(); mlflow.log_figure(fig, "plots/clusters_vs_labels.png"); plt.close(fig)
            mlflow.sklearn.log_model(
                sk_model=model, name="clustering_model",
                serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            )
            results.append({
                "run_name": run_name, "run_id": run.info.run_id,
                "feature_selection": feature_selection, "algorithm": algorithm,
                "dimensions": representation["output_dimensions"],
                "reduction_percent": 100 * (1 - representation["output_dimensions"] / representations["none"]["output_dimensions"]),
                **metrics,
            })
            print(f"{run_name}: dims={representation['output_dimensions']}, silhouette={metrics['silhouette_score']:.4f}, F1={metrics['mapped_f1']:.4f}")
results_df = pd.DataFrame(results).sort_values("mapped_f1", ascending=False).reset_index(drop=True)
results_df

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/08/04 17:38:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Combined_none_K-means: dims=146, silhouette=0.3200, F1=0.8752


2026/08/04 17:38:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/08/04 17:40:38 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


Combined_pca_AgglomerativeHierarchical: dims=29, silhouette=0.3488, F1=0.8628


2026/08/04 17:40:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Combined_pca_GaussianMixtureModel: dims=29, silhouette=0.2491, F1=0.5705


,run_name,run_id,feature_selection,algorithm,dimensions,reduction_percent,silhouette_score,davies_bouldin_index,calinski_harabasz_index,adjusted_rand_index,...,homogeneity,completeness,v_measure,mapped_accuracy,mapped_precision,mapped_recall,mapped_f1,cluster_count,noise_ratio,fit_seconds
0,Combined_laplacianscore_DBSCAN,335e0f4a2c8948f4ae00be1f6cb78738,laplacianscore,DBSCAN,48,67.123288,0.461723,1.717379,138.263639,0.378481,...,0.810277,0.268658,0.403522,0.94675,0.983578,0.900644,0.940286,34,0.08175,0.268106
1,Combined_variancethreshold_DBSCAN,48384526efe64d76af5ac392b9217dfa,variancethreshold,DBSCAN,114,21.917808,0.333572,1.333311,324.354806,0.465729,...,0.708207,0.304726,0.426108,0.92475,1.000000,0.838346,0.912065,26,0.07475,0.316832
2,Combined_none_DBSCAN,4b8ef19fb7114b11bca8e6b2a760875c,none,DBSCAN,146,0.000000,0.345289,1.971441,77.108537,0.442786,...,0.694684,0.291373,0.410549,0.92175,0.998071,0.833512,0.908399,25,0.07725,0.386623
3,Combined_correlationfiltering_DBSCAN,122a05237b5242b1abb3e34f0f324fa8,correlationfiltering,DBSCAN,125,14.383562,0.270237,1.958087,60.875225,0.438502,...,0.684636,0.288292,0.405734,0.92000,0.996778,0.830827,0.906268,24,0.08075,0.348879
4,Combined_pca_DBSCAN,4872307963bf49c9afa27cf9a28335bf,pca,DBSCAN,29,80.136986,0.424688,1.804888,87.231967,0.464443,...,0.689450,0.300899,0.418953,0.92000,0.997419,0.830290,0.906213,23,0.08025,0.248204
5,Combined_correlationfiltering_K-means,0d440e82d49b4be4b5d9c873ab0f720e,correlationfiltering,K-means,125,14.383562,0.237198,1.734033,337.419095,0.636609,...,0.575072,0.601157,0.587825,0.89900,0.987952,0.792696,0.879619,2,0.00000,0.147572
6,Combined_none_K-means,3a7b933f59ee4c9b8c60082c8a8ab394,none,K-means,146,0.000000,0.319958,1.516865,515.318550,0.626265,...,0.567276,0.594780,0.580702,0.89575,0.988506,0.785177,0.875187,2,0.00000,0.132482
7,Combined_pca_K-means,7c42628d398a4506a9c5a1a5114398e5,pca,K-means,29,80.136986,0.349627,1.442893,554.515933,0.626265,...,0.567276,0.594780,0.580702,0.89575,0.988506,0.785177,0.875187,2,0.00000,0.105088
8,Combined_variancethreshold_K-means,67c0c26e81c549f299fe4cbd279607b2,variancethreshold,K-means,114,21.917808,0.382750,1.283888,1754.851654,0.625474,...,0.565733,0.593042,0.579065,0.89550,0.987838,0.785177,0.874925,2,0.00000,0.144690
9,Combined_none_GaussianMixtureModel,898e2c8b8f7d4c12af60c5cfa42432ac,none,GaussianMixtureModel,146,0.000000,0.317807,1.509248,513.029541,0.625466,...,0.573466,0.603128,0.587923,0.89550,0.993169,0.780881,0.874324,2,0.00000,0.457270


In [6]:
baseline = results_df[results_df.feature_selection == "none"].set_index("algorithm")
comparison_df = results_df.copy()
comparison_df["silhouette_delta_vs_none"] = comparison_df.apply(lambda r: r.silhouette_score - baseline.loc[r.algorithm, "silhouette_score"], axis=1)
comparison_df["ari_delta_vs_none"] = comparison_df.apply(lambda r: r.adjusted_rand_index - baseline.loc[r.algorithm, "adjusted_rand_index"], axis=1)
comparison_df["f1_delta_vs_none"] = comparison_df.apply(lambda r: r.mapped_f1 - baseline.loc[r.algorithm, "mapped_f1"], axis=1)
best_method_per_algorithm = comparison_df.sort_values("mapped_f1", ascending=False).groupby("algorithm", as_index=False).first()[[
    "algorithm", "feature_selection", "dimensions", "silhouette_score",
    "adjusted_rand_index", "mapped_f1", "f1_delta_vs_none"
]]
selected_only = comparison_df[comparison_df.feature_selection != "none"]
best_representation = selected_only.sort_values(["mapped_f1", "silhouette_score"], ascending=False).iloc[0]
preservation_rows = []
for algorithm, group in comparison_df.groupby("algorithm"):
    base = baseline.loc[algorithm]
    preserved = group[(group.silhouette_score >= base.silhouette_score - 0.01) & (group.adjusted_rand_index >= base.adjusted_rand_index - 0.02) & (group.mapped_f1 >= base.mapped_f1 - 0.01)]
    if len(preserved): preservation_rows.append(preserved.sort_values("dimensions").iloc[0].to_dict())
minimum_structure_df = pd.DataFrame(preservation_rows)[["algorithm", "feature_selection", "dimensions", "reduction_percent", "silhouette_score", "adjusted_rand_index", "mapped_f1"]]
benefit_summary_df = comparison_df.groupby("feature_selection").agg(
    mean_silhouette_delta=("silhouette_delta_vs_none", "mean"),
    mean_ari_delta=("ari_delta_vs_none", "mean"),
    mean_f1_delta=("f1_delta_vs_none", "mean"),
    mean_dimensions=("dimensions", "mean"),
).reset_index().sort_values("mean_f1_delta", ascending=False)
display(Markdown("## Does feature selection improve clustering quality?")); display(comparison_df[["algorithm", "feature_selection", "dimensions", "silhouette_delta_vs_none", "ari_delta_vs_none", "f1_delta_vs_none"]])
display(Markdown("## Best representation for each clustering algorithm")); display(best_method_per_algorithm)
display(Markdown("## Smallest representation preserving intrinsic structure")); display(minimum_structure_df)
display(Markdown("## Average feature-selection benefit")); display(benefit_summary_df)

## Does feature selection improve clustering quality?

,algorithm,feature_selection,dimensions,silhouette_delta_vs_none,ari_delta_vs_none,f1_delta_vs_none
0,DBSCAN,laplacianscore,48,0.116433,-0.064305,0.031887
1,DBSCAN,variancethreshold,114,-0.011717,0.022943,0.003666
2,DBSCAN,none,146,0.000000,0.000000,0.000000
3,DBSCAN,correlationfiltering,125,-0.075053,-0.004284,-0.002131
4,DBSCAN,pca,29,0.079399,0.021657,-0.002186
5,K-means,correlationfiltering,125,-0.082760,0.010344,0.004432
6,K-means,none,146,0.000000,0.000000,0.000000
7,K-means,pca,29,0.029669,0.000000,0.000000
8,K-means,variancethreshold,114,0.062792,-0.000791,-0.000262
9,GaussianMixtureModel,none,146,0.000000,0.000000,0.000000


## Best representation for each clustering algorithm

,algorithm,feature_selection,dimensions,silhouette_score,adjusted_rand_index,mapped_f1,f1_delta_vs_none
0,AgglomerativeHierarchical,variancethreshold,114,0.381183,0.588855,0.863195,0.000242
1,DBSCAN,laplacianscore,48,0.461723,0.378481,0.940286,0.031887
2,GaussianMixtureModel,none,146,0.317807,0.625466,0.874324,0.000000
3,K-means,correlationfiltering,125,0.237198,0.636609,0.879619,0.004432


## Smallest representation preserving intrinsic structure

,algorithm,feature_selection,dimensions,reduction_percent,silhouette_score,adjusted_rand_index,mapped_f1
0,AgglomerativeHierarchical,pca,29,80.136986,0.348768,0.587321,0.862768
1,DBSCAN,pca,29,80.136986,0.424688,0.464443,0.906213
2,GaussianMixtureModel,variancethreshold,114,21.917808,0.378407,0.606599,0.865735
3,K-means,pca,29,80.136986,0.349627,0.626265,0.875187


## Average feature-selection benefit

,feature_selection,mean_silhouette_delta,mean_ari_delta,mean_f1_delta,mean_dimensions
2,none,0.000000,0.000000,0.000000,146.0
4,variancethreshold,0.043334,0.000822,-0.001236,114.0
0,correlationfiltering,-0.085124,-0.007676,-0.004946,125.0
1,laplacianscore,0.155606,-0.058795,-0.006051,48.0
3,pca,0.017407,-0.151399,-0.076536,29.0


# Conclusions from the executed experiment

## Does feature selection improve clustering quality?

Yes, but the improvement depends on the clustering algorithm and the evaluation criterion. Variance Threshold is the most reliable general-purpose selector: averaged across all four algorithms it raises Silhouette by 0.0433 and ARI by 0.0008 while reducing the encoded representation from 146 to 114 dimensions; its mean mapped-F1 change is only -0.0012. Laplacian Score gives the largest separation gain (mean Silhouette +0.1556), but its average ARI and F1 decline, so it is not uniformly better.

## Which feature-selection representation is best?

For the single best clustering result, **Laplacian Score + DBSCAN** is the winner: mapped F1 = **0.9403**, accuracy = **0.9468**, and Silhouette = **0.4617** using 48 encoded dimensions (67.1% fewer than the full 146-dimensional representation). For a selector intended to work consistently across different clusterers, **Variance Threshold** is the safer overall choice because it improves average internal separation without materially changing average ARI or F1.

## Which algorithms benefit most?

- **DBSCAN benefits most.** Laplacian Score improves F1 from 0.9084 to 0.9403 and Silhouette from 0.3453 to 0.4617, although ARI falls from 0.4428 to 0.3785. Variance Threshold gives a smaller but more balanced DBSCAN improvement: F1 0.9121 and ARI 0.4657.
- **K-Means benefits slightly from Correlation Filtering** in label agreement (F1 0.8796 versus 0.8752), while PCA preserves its F1 and raises Silhouette.
- **Agglomerative clustering benefits slightly from Variance Threshold** (F1 0.8632 versus 0.8630; Silhouette 0.3812 versus 0.3195).
- **Gaussian Mixture does not benefit in mapped F1.** The full representation remains best, and PCA is especially unsuitable for this configuration (F1 falls to 0.5705).

## How many dimensions preserve the intrinsic structure?

Using the notebook's preservation tolerances (no more than 0.01 loss in Silhouette, 0.02 in ARI, and 0.01 in F1), **29 PCA components** preserve the measured structure for K-Means, DBSCAN, and Agglomerative clustering, an 80.1% reduction from 146 encoded dimensions. Gaussian Mixture needs the 114-dimensional Variance Threshold representation to remain within those tolerances. This is a representation-dimension result; PCA components are not equivalent to 29 directly interpretable raw features.

## Is the selected space better for anomaly and hybrid IDS models?

The evidence is promising rather than universal. DBSCAN on Laplacian features provides the strongest attack-mapped F1 and separation, making that representation the best anomaly/hybrid candidate in this experiment. However, its lower ARI means the denser cluster structure does not align with the binary labels on every criterion. For a conservative hybrid pipeline, Variance Threshold + DBSCAN is the balanced alternative because it improves F1, ARI, and dimensionality together. Labels were used only for evaluation and for fitting the train-set cluster-to-label mapping, never to select features or fit clusters.